In [14]:

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os
from dotenv import load_dotenv

env = load_dotenv(dotenv_path='/mnt/d/linux/project/vars/.env')

#env vars for postgresql connection
mssqlsource = os.getenv('MSSQLSOURCE')
mssqluser = os.getenv('MSSQL_SPARK_USER')
mssqlpwd = os.getenv('MSSQL_SPARK_PWD')
mssqldriver = os.getenv('MSSQL_DRIVER')
sqlserverjar = os.getenv('SQLSERVER_JAR')
mssql_dwdest = os.getenv('MSSQL_CONN')

In [15]:
#CREATE SESSION
spark = ((
        SparkSession
            .builder
            .appName('dim_stock')
            .config('spark.jars', f'{sqlserverjar}')
            .config('spark.driver.extraClassPath', f'{sqlserverjar}')
            .config('spark.executor.extraClassPath', f'{sqlserverjar}')
        .getOrCreate()
))

spark.sparkContext.setLogLevel("WARN")

In [16]:
#first of all, lets create a dataframe from data table
stockitems_frame = (( spark.read
                        .format('jdbc')
                        .option('url', mssqlsource)
                        .option('user', mssqluser)
                        .option('password', mssqlpwd)
                        .option('driver', mssqldriver)
                        .option('query', 
                                
                                """  
                                select 

                                        wsi.StockItemID,
                                        wsi.StockItemName,
                                        wsi.SupplierID,
                                        sisg.stockgroupid,
                                        wsi.ColorID,
                                        wsi.UnitPackageID,
                                        wsi.OuterPackageID,
                                        wsi.QuantityPerOuter,
                                        wsi.Brand,
                                        wsi."Size",
                                        wsi.LeadTimeDays,
                                        wsi.IsChillerStock,
                                        wsi.TaxRate,
                                        wsi.UnitPrice,
                                        wsi.RecommendedRetailPrice,
                                        wsi.TypicalWeightPerUnit,
                                        wsi.MarketingComments,
                                        wsi.CustomFields,
                                        wsi.tags,
                                        wsi.ValidFrom,
                                        wsi.ValidTo
                                        
                                from Warehouse.StockItems as wsi

                                inner join

                                        Warehouse.StockItems_Archive as wsia

                                on wsi.StockItemID = wsia.StockItemID

                                inner join

                                        warehouse.stockitemstockgroups sisg

                                on wsi.stockitemid = sisg.stockitemid
                                
                                """      
                        ))).load()

stockitems_frame.show(10, False)

+-----------+----------------------------+----------+------------+-------+-------------+--------------+----------------+-----+----+------------+--------------+-------+---------+----------------------+--------------------+----------------------------+------------------------------------------------------------+---------------+-------------------+--------------------------+
|StockItemID|StockItemName               |SupplierID|stockgroupid|ColorID|UnitPackageID|OuterPackageID|QuantityPerOuter|Brand|Size|LeadTimeDays|IsChillerStock|TaxRate|UnitPrice|RecommendedRetailPrice|TypicalWeightPerUnit|MarketingComments           |CustomFields                                                |tags           |ValidFrom          |ValidTo                   |
+-----------+----------------------------+----------+------------+-------+-------------+--------------+----------------+-----+----+------------+--------------+-------+---------+----------------------+--------------------+----------------------------+

In [17]:
#cleaning customfield column wiht reg_exp
#remove characteres like " {} [] "
#renaming resulting columns
#drop columns that not satisfy our dataset

tempframe = ((stockitems_frame
                        .select('*',
                                f.regexp_replace('customfields', "[^0-9a-zA-Z , :]", "")
                                        .alias('temp_customfield'),
                                f.regexp_replace('tags', '[^0-9a-zA-Z]', '')
                                        .alias('tags'))
                        .withColumnRenamed('Size', 'size')
                        .drop('customfields')
                        .drop('tags')))
                


#now, select substring from temp_customfield
#leaving information about country manufacture
#droping unused columns

((
tempframe
        .select('*',
                f.substring_index('temp_customfield', ',', 1)
                        .alias('temp_country'),
                f.substring_index('temp_country', ':', -1)
                        .alias('countrymanufacture'))
        .drop('temp_country')
        
))

DataFrame[StockItemID: int, StockItemName: string, SupplierID: int, stockgroupid: int, ColorID: int, UnitPackageID: int, OuterPackageID: int, QuantityPerOuter: int, Brand: string, size: string, LeadTimeDays: int, IsChillerStock: boolean, TaxRate: decimal(18,3), UnitPrice: decimal(18,2), RecommendedRetailPrice: decimal(18,2), TypicalWeightPerUnit: decimal(18,3), MarketingComments: string, ValidFrom: timestamp, ValidTo: timestamp, temp_customfield: string, countrymanufacture: string]

In [18]:
#fill null values
#renaming airbyte columns
dimstockitem_frame = ((
        tempframe
            .fillna({'size': 'Unknown',
                    'colorid': 999,
                    'brand': 'Unknown',
                    'marketingcomments': 'Unknown'
                    })
            
        .drop('temp_customfield')
))

In [19]:
((
dimstockitem_frame
        .write
            .format('jdbc')
            .option('url', mssql_dwdest)
            .option('user', mssqluser)
            .option('password', mssqlpwd)
            .option('driver', mssqldriver)
            .option('mode', 'overwrite')
            .option('dbtable', 'dimension.product')
        .save(mode = 'overwrite')
        
))

In [20]:
spark.stop()